In [2]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from sklearn.ensemble import RandomForestClassifier
from sklearn.neighbors import KNeighborsClassifier
from sklearn.svm import SVC
from sklearn.model_selection import cross_val_score
from sklearn.model_selection import train_test_split
from sklearn import metrics
from sklearn.tree import DecisionTreeClassifier

import koreanize_matplotlib
import warnings
warnings.filterwarnings('ignore')

In [3]:
df = pd.read_csv('../Data/당뇨_전처리.csv')
df

,임신횟수,혈당,혈압,피부두께,인슐린,BMI,가족력지표,나이,당뇨
0,6,148.0,72.0,35.0,159.0,33.6,0.627,50,1
1,1,85.0,66.0,29.0,95.0,26.6,0.351,31,0
2,8,183.0,64.0,32.0,159.0,23.3,0.672,32,1
3,1,89.0,66.0,23.0,94.0,28.1,0.167,21,0
4,0,137.0,40.0,35.0,168.0,43.1,2.288,33,1
...,...,...,...,...,...,...,...,...,...
712,10,101.0,76.0,48.0,180.0,32.9,0.171,63,0
713,2,122.0,70.0,27.0,95.0,36.8,0.340,27,0
714,5,121.0,72.0,23.0,112.0,26.2,0.245,30,0
715,1,126.0,60.0,32.0,159.0,30.1,0.349,47,1


-----
-----
### SVM

In [ ]:
from sklearn.svm import SVC
from sklearn.model_selection import GridSearchCV
from sklearn.metrics import classification_report, confusion_matrix

In [15]:
# 앱으로 만들어야 하므로 유저가 쉽게 입력 할 수 있는 것들만 피쳐로 선정해서 해보는 것으로 함
df = df[['BMI','나이','가족력지표','피부두께', '혈당','당뇨']]
data = df.iloc[:,:-1]
target = df.iloc[:,-1]

# 파생변수 생성 (앱 입력/계산용)
data['Waist_Risk_Factor'] = (data['BMI'] ** 1.2) / 10
data['Abdominal_Glucose_Risk'] = (data['혈당'] * data['Waist_Risk_Factor']) / 100

# Log10대신 Log1p권장 (0값 방지)
data['가족력로그'] = np.log1p(data['가족력지표'])

# 중간 계산용/대체용 컬럼 정리
data.drop(['Waist_Risk_Factor','가족력지표'],axis = 1,inplace=True)

train_data, test_data, train_target, test_target = train_test_split(
    data,
    target,
    random_state=42,
    test_size= 0.2,
    stratify=df.iloc[:,-1]
)

from sklearn.preprocessing import StandardScaler

ss = StandardScaler()
train_scaled = ss.fit_transform(train_data)
test_scaled = ss.transform(test_data)


svc_parameters = [
    {
        'kernel': ['rbf'],
        'gamma': [0.0001, 0.001, 0.01, 0.1, 1],
        'C': [0.1, 1, 10, 100, 1000]
    },
    {
        'kernel': ['linear'],
        'C': [0.1, 1, 10, 100, 1000]
    }
]

clf = GridSearchCV(
    SVC(random_state=42),
    svc_parameters,
    cv=5,
    scoring='accuracy',
    n_jobs=1   # 네 환경에서는 1 권장 (한글 경로 이슈)
)

clf.fit(train_scaled, train_target)

print("최적의 파라미터 :", clf.best_params_)
print("최고 CV 점수 :", f"{clf.best_score_:.4f}")

# 8) 최종 평가
best_svc = clf.best_estimator_

print("-" * 30)
print("최종 Train 점수 :", f"{best_svc.score(train_scaled, train_target):.4f}")
print("최종 Test 점수 :", f"{best_svc.score(test_scaled, test_target):.4f}")

# 9) 상세 평가 (당뇨 모델에서는 추천)
pred = best_svc.predict(test_scaled)

print("\n[Confusion Matrix]")
print(confusion_matrix(test_target, pred))

print("\n[Classification Report]")
print(classification_report(test_target, pred))


최적의 파라미터 : {'C': 1, 'gamma': 0.1, 'kernel': 'rbf'}
최고 CV 점수 : 0.7924
------------------------------
최종 Train 점수 : 0.8063
최종 Test 점수 : 0.7986

[Confusion Matrix]
[[87  9]
 [20 28]]

[Classification Report]
              precision    recall  f1-score   support

           0       0.81      0.91      0.86        96
           1       0.76      0.58      0.66        48

    accuracy                           0.80       144
   macro avg       0.78      0.74      0.76       144
weighted avg       0.79      0.80      0.79       144



>모델은 과적합 없이 안정적으로 학습되었지만, 컬럼 축소 및 추정 변수 사용으로 인해 당뇨 클래스 재현율이 크게 감소했다.
앱 입력 편의성은 좋아졌지만, 실제 당뇨 환자를 놓치는 비율(FN)이 증가하여 의료/선별 목적에서는 보완이 필요하다.